# Config

In [1]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

In [2]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [3]:
import pandas as pd
import os
from preprocess.preprocess import clean_text, check_deleted_expressions
from preprocess.translate import translator, gen_text_for_embedding, final_clean, detect_language
import time
import json
import numpy as np

# Preparación de texto

In [18]:
# 1) Cargar datos
path = "/tmp/ocde/new_data"

#Cargar titulo, keys, etc.
filePATH = os.path.join(path, "old_data.xlsx")
df_data = pd.read_excel(filePATH, usecols=["Código VRID", "Título", "Keywords", "Resumen", "Depto Persona",
                                           "Facultad del Proyecto", "Interdisciplinario", "Transdisciplinario"])

#Cargar desafíos
filePATH = os.path.join(path, "new_data.xlsx")
#Pagina 0
df_ocde0 = pd.read_excel(os.path.join(path, filePATH), usecols=["Código VRID", "Categoria Disciplina", 
                                             "Sub Area Disciplina", "Tipo Área Disciplina"], sheet_name=2)
#Pagina 1
df_ocde1 =  pd.read_excel(os.path.join(path, filePATH), usecols=["Código VRID", "Categoria Disciplina", 
                                             "Sub Area Disciplina", "Tipo Área Disciplina"], sheet_name=3)

df_labels = pd.concat([df_ocde0, df_ocde1])


#Codes unique
codes_unique = df_labels["Código VRID"].astype(str).unique()


In [19]:
df_labels["Tipo Área Disciplina"].value_counts()

OCDE    591
ODS      54
Name: Tipo Área Disciplina, dtype: int64

Agregar area y subarea OCDE

In [20]:
df_data["Código VRID"] = df_data["Código VRID"].astype(str)
df_labels["Código VRID"] = df_labels["Código VRID"].astype(str)


# Merge en base a "Código VRID"
df_merged = pd.merge(
    df_data,
    df_labels,
    on="Código VRID",       # columna clave
    how="left"             # inner = solo los que coinciden en ambos
)

#save dataframe
print(df_merged.shape)
df_merged.head(2)

(1103, 11)


,Código VRID,Interdisciplinario,Transdisciplinario,Título,Keywords,Resumen,Facultad del Proyecto,Depto Persona,Categoria Disciplina,Sub Area Disciplina,Tipo Área Disciplina
0,2023-059,NO,NaN,PROGRAMA DE INVESTIGACIÓN EN ECONOMÍA DE RECUR...,"ECONOMÍA AMBIENTAL, ECONOMÍA DE RECURSOS NATUR...",NENRE EFD-CHILE EN UN PROGRAMA DE INVESTIGACIÓ...,CAMPUS CHILLÁN,"DEPARTAMENTO DE ECONOMÍA, DIRECCIÓN DE DESARRO...",NaN,NaN,NaN
1,40044450,SI,NaN,DISTRITO INNOVACIÓN ÑUBLE: HERRAMIENTA DE DESA...,"INNOVACIÓN, INVESTIGACIÓN Y DESARROLLO, AGRICU...",EL DISTRITO DE INNOVACIÓN DE ÑUBLE ES UN PROYE...,FACULTAD DE CIENCIAS VETERINARIAS,"DEPARTAMENTO DE CIENCIA ANIMAL, DEPARTAMENTO D...",NaN,NaN,NaN


In [21]:
df_merged["Tipo Área Disciplina"].value_counts()

OCDE    429
Name: Tipo Área Disciplina, dtype: int64

Agregar Desafíos país

In [10]:
# 1) Cargar datos
path = "/tmp/desafios/new_data"
filePATH = os.path.join(path, "Datos_DP.xlsx")
df_labels = pd.read_excel(filePATH, usecols=["Código VRID", "Desafío País"])

# Merge en base a "Código VRID"
df_merged_final = pd.merge(
    df_merged,
    df_labels,
    on="Código VRID",       # columna clave
    how="left"             # inner = solo los que coinciden en ambos
)

#save dataframe
print(df_merged_final.shape)
df_merged_final.head(2)

(1103, 12)


,Código VRID,Interdisciplinario,Transdisciplinario,Título,Keywords,Resumen,Facultad del Proyecto,Depto Persona,Categoria Disciplina,Sub Area Disciplina,Tipo Área Disciplina,Desafío País
0,2023-059,NO,NaN,PROGRAMA DE INVESTIGACIÓN EN ECONOMÍA DE RECUR...,"ECONOMÍA AMBIENTAL, ECONOMÍA DE RECURSOS NATUR...",NENRE EFD-CHILE EN UN PROGRAMA DE INVESTIGACIÓ...,CAMPUS CHILLÁN,"DEPARTAMENTO DE ECONOMÍA, DIRECCIÓN DE DESARRO...",NaN,NaN,NaN,3
1,40044450,SI,NaN,DISTRITO INNOVACIÓN ÑUBLE: HERRAMIENTA DE DESA...,"INNOVACIÓN, INVESTIGACIÓN Y DESARROLLO, AGRICU...",EL DISTRITO DE INNOVACIÓN DE ÑUBLE ES UN PROYE...,FACULTAD DE CIENCIAS VETERINARIAS,"DEPARTAMENTO DE CIENCIA ANIMAL, DEPARTAMENTO D...",NaN,NaN,NaN,NaN
